# External 327-image tri-class audit — frozen IQ-OTH GoogLeNet + PCA + SVM

**Exploratory dataset-source transfer test only.** Dataset has 117 Benign, 100 Malignant, and 110 Normal JPEG files. Patient identifiers and diagnostic provenance are not supplied in the uploaded archive. Results are **image-level**, not patient-independent. **No retraining, no hyperparameter tuning, no threshold adjustment** on these test images.

Add **two inputs** to Kaggle: the uploaded `archive (2).zip` dataset (or extracted BENIGN/MALIGNANT/NORMAL folders), and `GHDNN_FROZEN_MODEL.zip`. Enable GPU T4 and Internet (for the official ImageNet GoogLeNet weights), then run cells in order.


In [ ]:
%pip -q install "scikit-learn==1.6.1" pydicom
import numpy as np, sklearn, torch, torchvision
print('Versions:',torch.__version__,torchvision.__version__,sklearn.__version__)
print('GPU:',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# CELL 2 — Find mounted dataset and frozen model (works for ZIP or extracted folders)
from pathlib import Path
import zipfile, io, re, collections, json, hashlib, time
from PIL import Image
import numpy as np, cv2, torch, joblib, pandas as pd
from torchvision.models import googlenet, GoogLeNet_Weights
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score

ROOT = Path('/kaggle/input'); WORK = Path('/kaggle/working/EXTERNAL_327_RESULTS');WORK.mkdir(exist_ok=True)
ORDER = ['Benign','Malignant','Normal']

def is_class_path(path):
    bits=path.replace('\\','/').split('/')
    return len(bits)>=3 and bits[-1].lower().endswith(('.jpg','.jpeg','.png')) and bits[-2].upper()==bits[-3].upper() and bits[-2].upper() in {'BENIGN','MALIGNANT','NORMAL'}

zip_candidates=[];extracted=[]
for item in ROOT.rglob('*'):
    if item.is_file():
        if item.suffix.lower()=='.zip':
            try:
                with zipfile.ZipFile(item) as z:
                    names=[n for n in z.namelist() if is_class_path(n)]
                    if len(names)>200:zip_candidates.append((item,names))
            except (zipfile.BadZipFile,OSError):pass
        elif item.suffix.lower() in {'.jpg','.jpeg','.png'} and is_class_path(str(item)):
            extracted.append(item)

if zip_candidates:
    source_mode='zip'; DATA_ZIP=zip_candidates[0][0]; image_names=sorted(zip_candidates[0][1]);print('DATASET ZIP:', DATA_ZIP)
elif extracted:
    source_mode='folders'; image_names=sorted(extracted);print('DATASET EXTRACTED: JPEGs',len(image_names))
else:
    raise FileNotFoundError('Cannot find archive (2).zip or extracted BENIGN/BENIGN, MALIGNANT/MALIGNANT, NORMAL/NORMAL')

counter=collections.Counter((str(n).split('/')[-3] if source_mode=='zip' else n.parent.name).upper() for n in image_names)
print('COUNTS:',dict(counter),'TOTAL:',len(image_names))
assert len(image_names)==327 and dict(counter)=={'BENIGN':117,'MALIGNANT':100,'NORMAL':110},'Unexpected dataset composition. Stop and inspect inputs.'

frozen_matches=list(ROOT.rglob('GHDNN_FROZEN_MODEL.zip'))+list(ROOT.rglob('GHDNN_FROZEN_MODEL_REPACKED.zip'))
# Kaggle may unpack the ZIP and expose the joblib file directly
pipeline_files=list(ROOT.rglob('ghdnn_frozen_pipeline.joblib'))
assert frozen_matches or pipeline_files, 'Missing GHDNN_FROZEN_MODEL.zip input'
print('FROZEN MODEL:',frozen_matches[:1] or pipeline_files[:1]);print('READY: execute next cell')


In [ ]:
# CELL 3 — Exactly the frozen 64-patch IQ-OTH inference pipeline, image-level test
import sklearn
assert sklearn.__version__=='1.6.1',f'scikit-learn mismatch: {sklearn.__version__}, required 1.6.1'
if frozen_matches:
    with zipfile.ZipFile(frozen_matches[0]) as z:
        pipe=joblib.load(io.BytesIO(z.read('ghdnn_frozen_pipeline.joblib')))
else:
    pipe=joblib.load(pipeline_files[0])
print('Stored classes:',pipe['classes'],'SVM classes:',pipe['svm'].classes_)
assert list(pipe['classes']) == ORDER

# torchvision downloads official GoogLeNet IMAGENET1K_V1 once. Must have Kaggle Internet enabled.
model=googlenet(weights=GoogLeNet_Weights.IMAGENET1K_V1)
model.fc=torch.nn.Identity();model.eval()
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
mu=torch.tensor([.485,.456,.406],device=device).view(1,3,1,1)
sigma=torch.tensor([.229,.224,.225],device=device).view(1,3,1,1)
print('EXTRACTOR READY',device)

results=[]
if source_mode=='zip': zdata=zipfile.ZipFile(DATA_ZIP)
else:zdata=None
start=time.time()
for i,name in enumerate(image_names):
    path=str(name)
    data=zdata.read(name) if zdata is not None else Path(name).read_bytes()
    img=cv2.imdecode(np.frombuffer(data,dtype=np.uint8),cv2.IMREAD_GRAYSCALE)
    if img is None:raise ValueError(f'Unreadable image: {name}')
    frame=cv2.resize(img,(224,224),interpolation=cv2.INTER_AREA)
    patches=np.stack([cv2.resize(frame[r:r+28,c:c+28],(224,224),interpolation=cv2.INTER_LINEAR)
                      for r in range(0,224,28) for c in range(0,224,28)])
    feats=np.empty((64,1024),dtype=np.float32)
    for k in range(0,64,16):
        x=torch.from_numpy(patches[k:k+16,None].astype(np.float32)/255).to(device).repeat(1,3,1,1)
        x=(x-mu)/sigma
        with torch.inference_mode(): feat=model(x).cpu().numpy()
        feats[k:k+len(feat)]=feat
    v=pipe['scaler'].transform(pipe['pca'].transform(feats).reshape(1,-1))
    prediction=pipe['classes'][int(pipe['svm'].predict(v)[0])]
    raw=pipe['svm'].predict_proba(v)[0]
    prob_map={pipe['classes'][int(cls)]:float(raw[j]) for j,cls in enumerate(pipe['svm'].classes_)}
    true=next((cl for cl in ORDER if ('/'+cl.upper()+'/') in '/'+path.upper()+'/'),None)
    # Correct class always appears twice immediately above filename.
    true=path.replace('\\','/').split('/')[-2].title()
    results.append({'image':path,'truth':true,'prediction':prediction,'correct':int(true==prediction),
                    'prob_Benign':prob_map['Benign'],'prob_Malignant':prob_map['Malignant'],
                    'prob_Normal':prob_map['Normal'],'sha256':hashlib.sha256(data).hexdigest()})
    if (i+1)%25==0 or i+1==len(image_names): print(f'{i+1}/{len(image_names)} processed; elapsed {round(time.time()-start,1)} s',flush=True)
if zdata:zdata.close()
df=pd.DataFrame(results)
df.to_csv(WORK/'predictions_327.csv',index=False)
print('PREDICTIONS COMPLETE',len(df))


In [ ]:
# CELL 4 — Exact results, reproducibility report, small downloadable ZIP
from sklearn.metrics import accuracy_score,f1_score,balanced_accuracy_score,classification_report,confusion_matrix
import zipfile,json
cm=confusion_matrix(df['truth'],df['prediction'],labels=ORDER)
cm_df=pd.DataFrame(cm,index=['True_'+c for c in ORDER],columns=['Pred_'+c for c in ORDER])
cm_df.to_csv(WORK/'confusion_matrix.csv')
summary={'dataset_total_images':len(df),'true_class_counts':df['truth'].value_counts().to_dict(),
         'predicted_class_counts':df['prediction'].value_counts().to_dict(),
         'accuracy_pct':round(100*accuracy_score(df['truth'],df['prediction']),4),
         'macro_f1_pct':round(100*f1_score(df['truth'],df['prediction'],labels=ORDER,average='macro',zero_division=0),4),
         'balanced_accuracy_pct':round(100*balanced_accuracy_score(df['truth'],df['prediction']),4),
         'benign_recall_pct':round(100*classification_report(df['truth'],df['prediction'],labels=ORDER,output_dict=True,zero_division=0)['Benign']['recall'],4),
         'important_limitations':['This is image-level, not patient-independent validation',
                                  'Dataset source and class annotation provenance not established',
                                  'Do not tune the frozen model on this test set']}
(WORK/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
(WORK/'classification_report.txt').write_text(classification_report(df['truth'],df['prediction'],labels=ORDER,zero_division=0))
print('RESULTS:',json.dumps(summary,indent=2));print('CONFUSION MATRIX:');print(cm_df)
archive=Path('/kaggle/working/EXTERNAL_327_RESULTS.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for file in WORK.glob('*'):z.write(file,arcname=file.name)
print('DOWNLOAD:',archive)
